# 6차시 · 다음 단어를 맞히는 기계 (생성형 AI)

**탐구 질문**: 같은 질문을 해도 AI의 답이 매번 다른 이유는 무엇일까? 그럴듯한 거짓말(환각)은 왜 생길까?

오늘의 언어 모델은 ① 세기(학습) → ② 나누기(확률, 온도 적용) → ③ 뽑기(생성) 세 단계로 움직여요.

> 🔧 **규칙**: 맨 위 대문자 변수만, 한 번에 **하나씩** 바꾸고, 실행 **전에** 결과를 예측해 적어 보세요.

## 코드 실험실
아래 셀을 실행(Shift+Enter)해 보세요.

In [ ]:
# 6차시 · 다음 단어를 맞히는 기계 (아주 작은 언어 모델)
# 🔧 대문자 변수만 바꿔 가며 실험해 보세요. 한 번에 '하나만' 바꾸는 게 규칙!
TEMPERATURE = 1.0   # 온도: 작을수록 흔한 단어만, 클수록 드문 단어도 골라요
CONTEXT = 1         # 문맥 길이: 앞의 몇 단어를 보고 다음 단어를 고를까? (1, 2, 3)
LENGTH = 15         # 새로 만들 단어 수
START = "나는"       # 첫 단어 (TEXT 안에 있는 단어여야 해요)
SEED = 7            # 주사위 번호: SEED가 같으면 결과도 같아요

import random

TEXT = """
월요일. 나는 학교에 간다. 나는 친구와 점심을 먹는다. 나는 도서관에 간다. 나는 책을 읽는다.
화요일. 나는 학교에 간다. 나는 친구와 코딩을 한다. 코딩은 재미있다.
수요일. 나는 학교에 간다. 나는 친구와 축구를 한다. 축구는 재미있다.
목요일. 나는 학교에 간다. 나는 수학을 공부한다. 수학은 어렵다.
금요일. 나는 학교에 간다. 나는 친구와 인공지능을 공부한다. 인공지능은 재미있다. 인공지능은 다음 단어를 예측한다.
토요일. 나는 집에 있다. 나는 책을 읽는다. 책은 재미있다.
일요일. 나는 집에 있다. 나는 코딩을 한다. 코딩은 어렵다.
"""
words = TEXT.split()

# ① 학습: "앞 단어(들) → 다음 단어"가 몇 번 나왔는지 세기
counts = {}
for i in range(len(words) - CONTEXT):
    key = tuple(words[i : i + CONTEXT])     # 앞 단어 묶음 (문맥)
    nxt = words[i + CONTEXT]               # 바로 다음 단어
    if key not in counts:
        counts[key] = {}
    counts[key][nxt] = counts[key].get(nxt, 0) + 1

# ② 횟수 → 확률 (온도 적용: 횟수를 1/온도 제곱한 뒤 비율로)
def next_probs(key):
    c = counts.get(key, {})
    weights = {w: n ** (1 / TEMPERATURE) for w, n in c.items()}
    total = sum(weights.values())
    return {w: v / total for w, v in weights.items()}

# ③ 첫 문맥 정하기 + 다음 단어 확률표 보여 주기
random.seed(SEED)
first = [k for k in counts if k[0] == START]
if not first:
    raise SystemExit(f"'{START}'(이)라는 단어가 TEXT에 없어요. START를 바꿔 보세요.")
out = list(first[0])

probs = next_probs(tuple(out[-CONTEXT:]))
print(f"[확률표] '{' '.join(out)}' 다음에 올 단어 (온도 {TEMPERATURE})")
for w, p in sorted(probs.items(), key=lambda x: -x[1]):
    print(f"  {p * 100:5.1f}%  {'█' * round(p * 30):<30} {w}")

# ④ 확률대로 한 단어씩 뽑아 문장 만들기
for _ in range(LENGTH):
    probs = next_probs(tuple(out[-CONTEXT:]))
    if not probs:                       # 이 문맥 다음 단어를 본 적이 없으면 멈춤
        break
    nxt = random.choices(list(probs), weights=list(probs.values()))[0]
    out.append(nxt)

print("\n[AI가 만든 글]")
print(" ", " ".join(out))

# ⑤ 원문을 얼마나 그대로 베꼈을까? (연속 3단어가 원문에 있는 비율)
tri_text = {tuple(words[i : i + 3]) for i in range(len(words) - 2)}
tri_out = [tuple(out[i : i + 3]) for i in range(len(out) - 2)]
copied = sum(t in tri_text for t in tri_out) / max(1, len(tri_out))
print(f"\n원문 베끼기 비율: {copied * 100:.0f}%  (문맥 종류 {len(counts)}개)")

## 탐구 미션
각 미션마다 위 셀의 변수를 바꿔 다시 실행하고, 이 칸을 더블클릭해 예측·결과·설명을 적어 보세요.

### M1. 온도만 바꾸기
`TEMPERATURE`를 `1.0 → 0.2 → 3.0` 으로 바꿔 세 번 실행 (SEED는 그대로!)

- **예측**: 
- **결과**: 
- **설명**: 

### M2. 주사위 번호(SEED) 바꾸기
`TEMPERATURE = 1.0`으로 두고 `SEED`만 `7 → 8 → 9`

- **예측**: 
- **결과**: 
- **설명**: 

### M3. 문맥 길이와 베끼기
`CONTEXT`를 `1 → 2 → 3`. 마지막 줄의 원문 베끼기 비율과 문맥 종류 수를 기록표에서 비교

- **예측**: 
- **결과**: 
- **설명**: 

### M4. 오류를 고쳐라: 온도 0
`TEMPERATURE = 0` 으로 실행하면 오류가 납니다. 오류 이름과 줄 번호를 읽고, 온도가 0이면 가장 많이 나온 단어를 고르도록 `next_probs` 함수를 고쳐 보세요.

- **예측**: 
- **결과**: 
- **설명**: 

### M5. 데이터를 바꾸면 모델이 바뀐다
`TEXT` 안에 "나는 게임을 한다." 같은 문장을 여러 번 추가해 실행해 보세요. "나는" 다음 확률표가 어떻게 바뀌나요?

- **예측**: 
- **결과**: 
- **설명**: 

## 나의 결론
오늘의 탐구 질문에 대한 나의 결론 (3문장 이상):